# 🛡️ RMS-AV: Kaggle Full-Scale Deepfake Detection Training Pipeline
### Fail-Safe & Auto-Resuming Multimodal Training on the LAV-DF Benchmark (136,304 Media Samples)

**Key Features in This Kaggle Pipeline:**
- 💾 **Bulletproof Crash Recovery:** Saves an emergency crash checkpoint (`lavdf_rmsav_emergency_crash.pth`) on ANY failure, interruption, or timeout.
- ⏱️ **Mid-Epoch Step Checkpoints:** Persists interim weights every 500 steps so progress is never lost even if the VM shuts down mid-epoch.
- 🔄 **Seamless Multi-Version Auto-Resume:** Auto-scans `/kaggle/input/` and `/kaggle/working/` to resume from previous notebook runs.
- ⚡ **12-Hour Background Execution:** Fully compatible with Kaggle 'Save Version -> Save & Run All (Commit)' so it trains in the cloud without keeping your browser open.
- 🎓 **Publication-Ready Innovations:** Trains the Temporal Saliency Aggregator (TSA) + Cross-Modal Contrastive Incoherence (CM-FCI).

## 1. Verify Kaggle GPU Environment (Tesla T4 / P100)

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("[!] Warning: No GPU detected. In Kaggle Notebook Settings (right sidebar), set Accelerator to 'GPU T4 x2' or 'GPU P100'.")

## 2. Clone Latest Research Repository & Install Requirements

In [ ]:
import os
%cd /kaggle/working
REPO_URL = "https://github.com/akhildragon-07/DEEP_LEARNING.git"
REPO_DIR = "/kaggle/working/DEEP_LEARNING"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL
else:
    %cd $REPO_DIR
    !git pull origin main

%cd $REPO_DIR
!pip install -q -r requirements.txt
print("Repository synchronized and ready at:", os.getcwd())

## 3. Verify & Locate Official LAV-DF Dataset on Kaggle

In [ ]:
import os

# Check /kaggle/input for attached LAV-DF dataset
meta_found = None
media_root = None

if os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        for target in ["metadata.min.json", "metadata.json"]:
            if target in files:
                meta_found = os.path.join(root, target)
                media_root = root
                break
        if meta_found:
            break

if meta_found:
    print(f"[SUCCESS] LAV-DF Metadata located: {meta_found}")
    print(f"[SUCCESS] LAV-DF Media root located: {media_root}")
else:
    print("[!] Dataset not found in /kaggle/input. Downloading via kagglehub...")
    !pip install -q kagglehub
    import kagglehub
    path = kagglehub.dataset_download("elin75/localized-audio-visual-deepfake-dataset-lav-df")
    print("Downloaded to:", path)
    for root, dirs, files in os.walk(path):
        for target in ["metadata.min.json", "metadata.json"]:
            if target in files:
                meta_found = os.path.join(root, target)
                media_root = root
                break
        if meta_found:
            break
    print(f"Metadata located: {meta_found}")

## 4. Run Architectural Smoke Test (10/10 Checks)
*Validates Temporal Saliency Aggregator (TSA), CM-FCI, Dynamic Reliability Gating, and Backprop on 122 parameter tensors.*

In [ ]:
%cd /kaggle/working/DEEP_LEARNING
!python -m lavdf_pipeline.smoke_test

## 5. Launch Training with Fail-Safe Auto-Resume & Periodic Step Checkpointing

**How Crash-Proofing Works:**
1. **Every 500 steps (~15 min):** An interim checkpoint is saved to `/kaggle/working/checkpoints_lavdf/lavdf_rmsav_latest.pth`.
2. **If an error or timeout happens:** An emergency crash checkpoint (`lavdf_rmsav_emergency_crash.pth`) is automatically captured.
3. **Auto-Resume:** Setting `--resume` automatically detects previous checkpoints (including from previous Kaggle notebook versions attached as input) and resumes without starting from scratch!

> **Tip for 100% Autonomous Training:** Click **Save Version** (top-right) -> Select **Save & Run All (Commit)** -> Kaggle will train in the background for up to 9-12 hours even if you close your computer.

In [ ]:
# Training Hyperparameters
EPOCHS = 10
BATCH_SIZE = 16
GRAD_ACCUM = 2              # Effective batch size = 32
LEARNING_RATE = 1e-4
SAVE_INTERVAL_STEPS = 500   # Fail-safe save every 500 batches
LIMIT_SAMPLES = None        # Set to None for full 78,703 clips; or e.g. 5000 for rapid test

cmd = [
    "python -m lavdf_pipeline.train",
    "--mode rmsav",
    f"--epochs {EPOCHS}",
    f"--batch_size {BATCH_SIZE}",
    f"--grad_accum_steps {GRAD_ACCUM}",
    f"--lr {LEARNING_RATE}",
    "--aux_weight 0.20",
    "--fci_weight 0.15",
    f"--save_interval_steps {SAVE_INTERVAL_STEPS}",
    "--save_dir /kaggle/working/checkpoints_lavdf",
    "--log_dir /kaggle/working/results_lavdf",
    "--resume",
]

if meta_found:
    cmd.append(f'--metadata_path "{meta_found}"')
    cmd.append(f'--media_root "{media_root}"')

if LIMIT_SAMPLES:
    cmd.append(f"--limit_samples {LIMIT_SAMPLES}")

full_cmd = " ".join(cmd)
print("Running Command:", full_cmd)
!{full_cmd}

## 6. Full Held-Out Evaluation & Modality-Drop Stress Testing
*Evaluates the best checkpoint on the official 26,100 held-out test split under Clean, Missing-Video, and Missing-Audio conditions.*

In [ ]:
!python -m lavdf_pipeline.evaluate \
    --mode rmsav \
    --checkpoint /kaggle/working/checkpoints_lavdf/lavdf_rmsav_best.pth \
    --stress_test \
    --save_dir /kaggle/working/results_lavdf

## 7. Generate Publication Figures (ROC Curves, Gate Distribution, Stress Retention)

In [ ]:
!python generate_paper_artifacts.py \
    --results_csv /kaggle/working/results_lavdf/lavdf_benchmark_comparison.csv \
    --output_dir /kaggle/working/paper/artifacts

from IPython.display import Image, display
import glob
for fig in sorted(glob.glob("/kaggle/working/paper/artifacts/*.png") + glob.glob("paper/figures/*.png")):
    print(f"Displaying: {fig}")
    display(Image(filename=fig, width=650))

## 8. Package Checkpoints & Results for 1-Click Download
*Produces a clean, compact zip file in `/kaggle/working/RMSAV_Trained_Models_and_Results.zip`.*

In [ ]:
!python prepare_deployable_checkpoints.py
!zip -r /kaggle/working/RMSAV_Trained_Models_and_Results.zip \
    /kaggle/working/checkpoints_lavdf \
    /kaggle/working/results_lavdf \
    /kaggle/working/paper/artifacts
print("\n[DONE] Package ready for 1-click download at: /kaggle/working/RMSAV_Trained_Models_and_Results.zip")